In [1]:
from tqdm.auto import tqdm
from pathlib import Path
import numpy as np

# import geopandas as gpd
import xarray as xr
# import xclim
import logging
logger = logging.getLogger(__name__)

# from ocab.config import Config
from ocab.basins.stats import read_data#, basin_statistics
# from ocab.utils.geography import compute_pixel_area

import warnings
warnings.filterwarnings('ignore', message='invalid value encountered in sqrt', category=RuntimeWarning)

## Configuration

In [2]:
# dataset configuration
# cfg = Config('../CAMELS/config_CAMELS_v200.yml')

# # basins shapefile
# basins_file = cfg.path_dataset / 'preprocessing' / 'basins' / 'output' / 'stations_basins_3sec.geojson'

# meteorology
path_data = Path('/home/casadoj/Data/')
meteo = 'ROCIO-IBEB'
zarr_store = f'{meteo}_1979-2022.zarr'
# pet_method = 'HG85' # 'HG85': Hargreaves, 'DA02': Droogers-Allen

# output
path_out = path_data / meteo / 'GIS'
path_out.mkdir(parents=False, exist_ok=True)

### Data

In [3]:
# load meteorological data
zarr_store = path_data / meteo / zarr_store
if zarr_store.is_dir():
    data = read_data(zarr_store)
    print(f"{data.nbytes / 1e9:.2f} GB")
    # rewrite temperature units
    for var in ['mintemp', 'maxtemp']:
        data[var].attrs['units'] = 'degC'
    # rewrite precipitation units
    data['precipitation'].attrs['units'] = 'mm/d'
else:
    logger.error(f"Zarr store doesn't exist: {zarr_store}")

12.96 GB


In [31]:
# compute average precipitation
precip_mean = data['precipitation'].mean('time').compute()

In [ ]:
# precip_mean.rio.to_raster(path_out / 'avg_precipitation.tif', compress='deflate')

In [24]:
data.rio.crs

CRS.from_wkt('GEOGCRS["undefined",BASEGEOGCRS["undefined",ENSEMBLE["World Geodetic System 1984 ensemble",MEMBER["World Geodetic System 1984 (Transit)",ID["EPSG",1166]],MEMBER["World Geodetic System 1984 (G730)",ID["EPSG",1152]],MEMBER["World Geodetic System 1984 (G873)",ID["EPSG",1153]],MEMBER["World Geodetic System 1984 (G1150)",ID["EPSG",1154]],MEMBER["World Geodetic System 1984 (G1674)",ID["EPSG",1155]],MEMBER["World Geodetic System 1984 (G1762)",ID["EPSG",1156]],MEMBER["World Geodetic System 1984 (G2139)",ID["EPSG",1309]],MEMBER["World Geodetic System 1984 (G2296)",ID["EPSG",1383]],ELLIPSOID["WGS 84",6378137,298.257223563,LENGTHUNIT["metre",1],ID["EPSG",7030]],ENSEMBLEACCURACY[2.0],ID["EPSG",6326]],PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8901]]],DERIVINGCONVERSION["Pole rotation (netCDF CF convention)",METHOD["Pole rotation (netCDF CF convention)"],PARAMETER["Grid north pole latitude (netCDF CF convention)",49.5,ANGLEUNIT["degree",0.0174532925199433,ID

In [ ]:
precip_mean.rio.crs

In [27]:
data

<xarray.Dataset> Size: 13GB
Dimensions:        (time: 16071, rlat: 240, rlon: 280)
Coordinates:
  * time           (time) datetime64[ns] 129kB 1979-01-01 ... 2022-12-31
  * rlat           (rlat) float64 2kB -6.45 -6.4 -6.35 -6.3 ... 5.4 5.45 5.5
  * rlon           (rlon) float64 2kB -5.0 -4.95 -4.9 -4.85 ... 8.85 8.9 8.95
    rotated_pole   int64 8B 0
Data variables:
    precipitation  (time, rlat, rlon) float32 4GB dask.array<chunksize=(365, 40, 40), meta=np.ndarray>
    mintemp        (time, rlat, rlon) float32 4GB dask.array<chunksize=(365, 40, 40), meta=np.ndarray>
    maxtemp        (time, rlat, rlon) float32 4GB dask.array<chunksize=(365, 40, 40), meta=np.ndarray>
    lon            (rlat, rlon) float32 269kB dask.array<chunksize=(40, 40), meta=np.ndarray>
    lat            (rlat, rlon) float32 269kB dask.array<chunksize=(40, 40), meta=np.ndarray>
Attributes:
    title:        AEMET High-resolution (0.05 deg) daily gridded maximum temp...
    institution:  Agencia Estatal de Meteorologia (AEMET, www.aemet.es)
    references:   Peral, C., Navascu�s, B., Ramos, P. Available at: http://ww...
    history:      Creation year 2019
    Conventions:  CF-1.7
    version:      1.0

In [ ]:
precip_mean

<xarray.DataArray 'precipitation' (rlat: 240, rlon: 280)> Size: 269kB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(240, 280), dtype=float32)
Coordinates:
  * rlat          (rlat) float64 2kB -6.45 -6.4 -6.35 -6.3 ... 5.35 5.4 5.45 5.5
  * rlon          (rlon) float64 2kB -5.0 -4.95 -4.9 -4.85 ... 8.8 8.85 8.9 8.95
    rotated_pole  int64 8B 0
Attributes:
    long_name:  precipitation amount
    units:      mm/d
    table:      1

In [ ]:
# 1. Select your variable and ensure spatial dimension names are y and x
da_precip = data['precipitation'].rename({'rlat': 'y', 'rlon': 'x'})


In [32]:

# 2. Assign the CRS directly from the original dataset
precip_mean.rio.write_crs(data.rio.crs, inplace=True)

# 3. Reproject to standard WGS84
precip_mean_wgs84 = precip_mean.rio.reproject("EPSG:4326")


In [33]:

# 4. Set nodata and export
precip_mean_wgs84.rio.write_nodata(np.nan, inplace=True)
precip_mean_wgs84.rio.to_raster(path_out / "avg_precipitation_wgs84.tif", compress="deflate")